In [2]:
#2. Which combination of production studio, genre, runtime
#and release timing best explains rating-based anime movie success in the last 26 years?

import os
import requests
import json
from datetime import datetime
from dotenv import load_dotenv
import time

load_dotenv('../../test.env')
api_key_TMDB = os.getenv("TMDB_API_KEY")
api_key_OMDB = os.getenv("OMDB_API_KEY1")

if not api_key_TMDB:
    print("TMDB API KEY not found check env file")
    exit()
else:
    print("TMDB KEY loaded")

if not api_key_OMDB:
    print("OMDB API KEY not found check env file")
    exit()
else:
    print("OMDB KEY loaded")




TMDB KEY loaded
OMDB KEY loaded


In [3]:
#Building our links for the different api
tmdb_base = "https://api.themoviedb.org/3"
tmbd_endpoint = "/discover/movie"  #Get all result for "" fitting criteria
tmdb_url = tmdb_base + tmbd_endpoint



In [5]:
def collect_anime_movies(apikey_tmdb, apikey_omdb):

    # Calculate 26 years ago
    current_year = datetime.now().year
    start_year = current_year - 40
    start_date = f"{start_year}-01-01"

    #Filter
    parameters = {
        'api_key' : apikey_tmdb,
        'with_genres' : 16,         #Genre Animation
        'with_original_language' : 'ja', #Language Japanese
        'primary_release_date.gte' : start_date,
        "vote_count.gte": 50,       #Skip over unimportant movies
        'sort_by' : 'primary_release_date.asc',
        'page' : 1 
    } 

    result = []
    total_pages = 1

    # Going through all pages
    while parameters['page'] <= total_pages:
        tmdb_response=requests.get(tmdb_url, params=parameters)

        # Send request to api
        if tmdb_response.status_code == 200:
            tmdb_data = tmdb_response.json()
            total_pages = tmdb_data['total_pages']

            #For each movie per page
            for tmdb_movie in tmdb_data['results']:
                score_list=[]
                imdb_score = 0.0
                meta_score = 0.0
                rotten_score = 0.0
                production_comps_list = []
                genres_list = []

                # request link to get details about each movie, for runtime, revenue, budget
                detail_url = f"https://api.themoviedb.org/3/movie/{tmdb_movie['id']}"
                detail_params = {
                    "api_key": apikey_tmdb
                }

                # sending request
                try:
                    detail_response = requests.get(detail_url, params=detail_params)
                    if detail_response.status_code == 200:
                        detail_data = detail_response.json()
                except:
                    pass

                # Creating a list with all production studios
                for studio in detail_data['production_companies']:
                    production_comps_list.append(studio['name'])

                for genr in detail_data['genres']:
                    genres_list.append(genr['name'])

                # request link to get ratings about each movie from omdb api
                omdb_url = f"http://www.omdbapi.com/?"
                omdb_params = {
                    'apikey' : apikey_omdb,
                    'i' : detail_data['imdb_id']
                }

                # sending request
                try:
                    omdb_response = requests.get(omdb_url, params=omdb_params)
                    if omdb_response.status_code == 200:
                        omdb_data = omdb_response.json()
                except:
                    pass
            
                # Gathering TMDb, IMDb, Metacritic and Rotten Tomatoes scores and calculating their mean, to get average movie score
                try:
                    tmdb_score = tmdb_movie['vote_average']
                    score_list.append(tmdb_score)
                except:
                    pass

                try:
                    rotten_score = omdb_data['Ratings'][1]['Value']

                    rotten_score = rotten_score[0:-1] # removing "%" from the end of the string to be able to convert string to float

                    rotten_score = ((float(rotten_score))/10)
                    score_list.append(rotten_score)
                except:
                    pass

                try:
                    meta_score = ((float(omdb_data['Metascore']))/10) # converting Metascore x/100 to decimal number x/10
                    score_list.append(meta_score)
                except:
                    pass

                try:
                    imdb_score = float(omdb_data['imdbRating'])
                    score_list.append(imdb_score)
                except:
                    pass

                avg_movie_score = sum(score_list)/len(score_list)

                #add movie to result
                result.append({
                    'title' : tmdb_movie['title'],
                    'release_year' : tmdb_movie['release_date'][0:4],
                    'genres' : genres_list,
                    'runtime' : detail_data['runtime'],
                    'production_studios' : production_comps_list,
                    'tmdb_id' : tmdb_movie['id'],                
                    'imdb_id' : detail_data['imdb_id'],
                    'tmdb_rating' : tmdb_movie['vote_average'],
                    'imdb_rating' : imdb_score,
                    'rotten_score' : rotten_score,
                    'meta_score' : meta_score,
                    'avg_score' : round(avg_movie_score, 2)
                })

                #Spam protection
                time.sleep(0.25)
            


            print(f"Fethed page {parameters['page']} of {total_pages}")

            #Going over to next page
            parameters['page'] += 1

            #Spam protection
            time.sleep(0.2)
    
        else:
            print(f"Error on page {parameters['page']}")
            break    

    print(f"loaded {len(result)} movies.\n")

    print("The first five movies on the list:")
    for movie in result[:5]:
        print(movie)

    return result

result = collect_anime_movies(api_key_TMDB,api_key_OMDB)


Fethed page 1 of 32
Fethed page 2 of 32
Fethed page 3 of 32
Fethed page 4 of 32
Fethed page 5 of 32
Fethed page 6 of 32
Fethed page 7 of 32
Fethed page 8 of 32
Fethed page 9 of 32
Fethed page 10 of 32
Fethed page 11 of 32
Fethed page 12 of 32
Fethed page 13 of 32
Fethed page 14 of 32
Fethed page 15 of 32
Fethed page 16 of 32
Fethed page 17 of 32
Fethed page 18 of 32
Fethed page 19 of 32
Fethed page 20 of 32
Fethed page 21 of 32
Fethed page 22 of 32
Fethed page 23 of 32
Fethed page 24 of 32
Fethed page 25 of 32
Fethed page 26 of 32
Fethed page 27 of 32
Fethed page 28 of 32
Fethed page 29 of 32
Fethed page 30 of 32
Fethed page 31 of 32
Fethed page 32 of 32
loaded 632 movies.

The first five movies on the list:
{'title': 'Fist of the North Star', 'release_year': '1986', 'genres': ['Animation', 'Action', 'Science Fiction'], 'runtime': 110, 'production_studios': ['Shueisha', 'Toei Animation'], 'tmdb_id': 19877, 'imdb_id': 'tt0142371', 'tmdb_rating': 6.786, 'imdb_rating': 7.1, 'rotten_score'

In [6]:
import csv
def save_to_csv(result):
    # file name
    filename = "details_anime_movies_2000.csv"

    print("write data in CSV")

    # give writing perm and add utf-8 for "Sonderzeichen"
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
    
        # define lines
        columns = ['id', 'tmdb_id', 'imdb_id', 'title', 'release_year', 'genres', 'runtime', 'production_studios', 'tmdb_rating', 'imdb_rating', 'rotten_score', 'meta_score', 'avg_score']
        writer = csv.DictWriter(file, fieldnames=columns)

        # write headline
        writer.writeheader()

        # add each row
        for index, movie in enumerate(result, start=1):
            writer.writerow({
                'id': index,
                'tmdb_id' : movie['tmdb_id'],
                'imdb_id' : movie['imdb_id'],
                'title': movie['title'],
                'release_year': movie['release_year'],
                'genres': movie['genres'],
                'runtime' : movie['runtime'],
                'production_studios' : movie['production_studios'],
                'tmdb_rating' : movie['tmdb_rating'],
                'imdb_rating' : movie['imdb_rating'],
                'rotten_score' : movie['rotten_score'],
                'meta_score' : movie['meta_score'],
                'avg_score' : movie['avg_score']
            })

    print(f"Finished writing '{filename}'")
    
save_to_csv(result)

write data in CSV
Finished writing 'details_anime_movies_2000.csv'
